In [1]:
%pip install -U "langchain[google-genai]"

/home/lakshay/LangChain/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
load_dotenv()

model = init_chat_model(model_provider="google_genai", 
                        model ="gemini-2.5-flash",
                        temperature=0.7,
                        max_output_tokens=1024,
                        top_p=0.95,
                        top_k=40,
                        timeout = 30,
                        max_retries = 3,
                        load_env=True)


# response = model.invoke("What is the capital of France?")
for chunk in model.stream("Why do parrots mimic human speech?"):
    print(chunk.content, end="", flush=True)

print()


In [ ]:
conversation = [
    {"role": "system", "content": "You are a helpful assistant that translates English to French."},
    {"role": "user", "content": "Translate: I love programming."},
    {"role": "assistant", "content": "J'adore la programmation."},
    {"role": "user", "content": "Translate: I love building applications."}
]

response = model.invoke(conversation)
print(response)  

In [ ]:
from langchain.messages import AIMessage, HumanMessage, SystemMessage

conversation = [
    SystemMessage(content="You are a helpful assistant that translates English to French."),
    HumanMessage(content="Translate: I love programming."),
    AIMessage(content="J'adore la programmation."),
    HumanMessage(content="Translate: I love building applications.")
]

response = model.invoke(conversation)
print(response)

In [ ]:
responses = model.batch([
    "What is the capital of germany?",
    "What is the capital of india",
    "what is the capital of france",
])

for response in responses:
    print(response)

In [13]:
for response in model.batch_as_completed([
    "Why do parrots have colorful feathers?",
    "How do airplanes fly?",
    "What is quantum computing?"
]):
    print(response)

(2, AIMessage(content="Quantum computing is a **new type of computing that harnesses the principles of quantum mechanics to perform calculations and solve complex problems that are intractable for classical computers.**\n\nHere's a breakdown of what that means:\n\n## Key Differences from", additional_kwargs={}, response_metadata={'finish_reason': 'MAX_TOKENS', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a09419-d2cc-7ac1-b3d7-ef1050065f49-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 6, 'output_tokens': 1020, 'total_tokens': 1026, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 974}}))
(0, AIMessage(content="Parrots have such vibrant and diverse feather colors due to a combination of evolutionary pressures and unique biological mechanisms. It's not just one reason, but several factors working together:\n\n1.  ", additional_kwargs={}, response_metadata={'finish_reason'

### Tool calling through model 

In [ ]:
from langchain.tools import tool

@tool
def get_current_weather(location: str) -> str:
    """Get the current weather in a given location"""
    return f"The current weather in {location} is sunny."

model_with_tool = model.bind_tools([get_current_weather])

response = model_with_tool.invoke("What is the current weather in New York?")
 
for tool_call in response.tool_calls:
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

print(response)

In [19]:
### Structured Output Example

from pydantic import BaseModel , Field

class Movie(BaseModel):
    """A movie with details"""

    title: str = Field(description="The title of the movie")
    year: int = Field(description="The year the movie was released")
    director: str = Field(description="The director of the movie")
    stars: list = Field(description="The main actors in the movie")
    rating: float = Field(description="The movie's rating out of 10")

model_with_structure = model.with_structured_output(Movie)
response = model_with_structure.invoke("Provide details about the movie Inception")
print(response)  

title='Inception' year=2010 director='Christopher Nolan' stars=['Leonardo DiCaprio', 'Joseph Gordon-Levitt', 'Elliot Page', 'Tom Hardy', 'Ken Watanabe'] rating=8.8


In [21]:
from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(model="gemini-2.5-flash",
                               temperature=0.7,
                               max_output_tokens=1024,
                               top_p=0.95,
                               top_k=40,
                               timeout=30,
                               max_retries=3)

response = model.invoke("In which movie iron man was kidnapped by terrorists?")
print(response)


content='That would be in the first **Iron Man** movie (2008).\n\nTony Stark is kidnapped by the Ten Rings terrorist group in Afghanistan after demonstrating his Jericho missile. While imprisoned, he is forced to build a weapon for them, but instead secretly constructs the first Iron Man suit (the Mark I) to escape.' additional_kwargs={} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--01a09452-c26d-7662-8c5f-eda65e0ea352-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 11, 'output_tokens': 239, 'total_tokens': 250, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 173}}
